## Testing Noramlizng Embeddings Before Inference

In [ ]:
from esmdmsfunctions import *

In [ ]:
unique_df = get_unique_df()

In [ ]:
embedding_array = np.array([x for x in unique_df["Embeddings"].to_list()])
embedding_array.shape
z_embeddings = np.zeros_like(embedding_array)
for layer in range(embedding_array.shape[1]):
    z_embeddings[:, layer, :] = z_normalize(embedding_array[:, layer, :])

# insert back into the dataframe
unique_df["Embeddings"] = [z_embeddings[i] for i in range(z_embeddings.shape[0])]

In [ ]:
plot_from_df(unique_df)
plot_from_df(shuffle_replicates(unique_df, replicates=[2], random_seed=20))
plot_from_df(shuffle_replicates(unique_df, replicates=[0, 1, 2], random_seed=20))

In [ ]:
no_norm_df = get_unique_df()
plot_from_df(no_norm_df)
plot_from_df(shuffle_replicates(no_norm_df, replicates=[2], random_seed=20))
plot_from_df(shuffle_replicates(no_norm_df, replicates=[0, 1, 2], random_seed=20))

# Normalizing across the other dimension of embeddings

In [ ]:
unique_df = get_unique_df()

embedding_array = np.array([x for x in unique_df["Embeddings"].to_list()])
embedding_array.shape
z_embeddings = np.zeros_like(embedding_array)
dimensions = embedding_array.shape[2]
for dim in range(dimensions):
    z_embeddings[:, :, dim] = z_normalize(embedding_array[:, :, dim])

# insert back into the dataframe
unique_df["Embeddings"] = [z_embeddings[i] for i in range(z_embeddings.shape[0])]

In [ ]:
plot_from_df(unique_df)
plot_from_df(shuffle_replicates(unique_df, replicates=[2], random_seed=20))
plot_from_df(shuffle_replicates(unique_df, replicates=[0, 1, 2], random_seed=20))

# Normalizing for a dimension in a given layer

probably makes the most sense?

In [ ]:
# By Layer normalization
embedding_array = np.array([x for x in unique_df["Embeddings"].to_list()])
embedding_array.shape
z_embeddings = np.zeros_like(embedding_array)
for layer in range(embedding_array.shape[1]):
    z_embeddings[:, layer, :] = z_normalize(embedding_array[:, layer, :])

# insert back into the dataframe
unique_df["Embeddings"] = [z_embeddings[i] for i in range(z_embeddings.shape[0])]


# By layer and dimension normalization
unique_df = get_unique_df()
embedding_array = np.array([x for x in unique_df["Embeddings"].to_list()])
embedding_array.shape
z_embeddings = np.zeros_like(embedding_array)
dimensions = embedding_array.shape[2]
layers = embedding_array.shape[1]
for dim in range(dimensions):
    for layer in range(layers):
        z_embeddings[:, layer, dim] = z_normalize(embedding_array[:, layer, dim])
# insert back into the dataframe
unique_df["Embeddings"] = [z_embeddings[i] for i in range(z_embeddings.shape[0])]

In [ ]:
plot_from_df(unique_df)
plot_from_df(shuffle_replicates(unique_df, replicates=[2], random_seed=20))
plot_from_df(shuffle_replicates(unique_df, replicates=[0, 1, 2], random_seed=20))

# BG505 and BF520 Comparison with Normalization

Let's make these replicate replicate consistency plots with bg505 AND bf520

In [ ]:
'''def get_unique_df(filepath=default_emb_path):
    """Get the protein dataframe from the embedding pickle file,
    combining all entries with the same protein sequence"""
    
    whole_df = pd.read_pickle(filepath)

    # Combine the prenums and postnums of any entries with the same potein sequence
    whole_df['PreNums'] = whole_df['PreNums'].apply(lambda x: np.array(x))
    whole_df['PostNums'] = whole_df['PostNums'].apply(lambda x: np.array(x))
    whole_df = whole_df.groupby('ProteinSequence').agg({
        'PreNums': lambda x: np.sum(x.tolist(), axis=0),
        'PostNums': lambda x: np.sum(x.tolist(), axis=0),
        'Embeddings': 'first'
    }).reset_index()

    whole_df = whole_df[whole_df["Embeddings"].notnull()].reset_index(drop=True)

    return whole_df'''
    
    
    
import pandas as pd
import numpy as np

def inverse_embedding_df_transfer(transformed_df: pd.DataFrame) -> pd.DataFrame:
    """
    Inverse of embedding_df_transfer_optimized.
    
    Converts a DataFrame with columns [Generation, Embedding, Frequency, Replicate]
    back to a DataFrame with columns [Embeddings, PreNums, PostNums].
    
    Parameters
    ----------
    transformed_df : pd.DataFrame
        DataFrame with columns: Generation, Embedding, Frequency, Replicate
        - Generation: 0 (pre-selection) or 1 (post-selection)
        - Embedding: the embedding vector
        - Frequency: count value
        - Replicate: 1-indexed replicate number
    
    Returns
    -------
    pd.DataFrame
        DataFrame with columns: Embeddings, PreNums, PostNums
        - Embeddings: list of unique embedding vectors
        - PreNums: 2D array of pre-selection counts (variants x replicates)
        - PostNums: 2D array of post-selection counts (variants x replicates)
    """
    if transformed_df.empty:
        return pd.DataFrame(columns=["Embeddings", "PreNums", "PostNums"])
    
    # Determine number of replicates
    num_reps = transformed_df["Replicate"].max()
    
    # Get unique embeddings while preserving order
    # Convert embeddings to tuples for hashing (if they're arrays/lists)
    def to_hashable(emb):
        if isinstance(emb, np.ndarray):
            return tuple(emb.tolist())
        elif isinstance(emb, list):
            return tuple(emb)
        return emb
    
    # Build mapping of embedding -> index, preserving first occurrence order
    embedding_to_idx = {}
    unique_embeddings = []
    
    for emb in transformed_df["Embedding"]:
        key = to_hashable(emb)
        if key not in embedding_to_idx:
            embedding_to_idx[key] = len(unique_embeddings)
            unique_embeddings.append(emb)
    
    num_variants = len(unique_embeddings)
    
    # Initialize count arrays with zeros
    pre_counts = np.zeros((num_variants, num_reps), dtype=int)
    post_counts = np.zeros((num_variants, num_reps), dtype=int)
    
    # Fill in the counts from the transformed DataFrame
    for _, row in transformed_df.iterrows():
        emb_key = to_hashable(row["Embedding"])
        var_idx = embedding_to_idx[emb_key]
        rep_idx = row["Replicate"] - 1  # Convert to 0-indexed
        
        if row["Generation"] == 0:
            pre_counts[var_idx, rep_idx] = row["Frequency"]
        else:  # Generation == 1
            post_counts[var_idx, rep_idx] = row["Frequency"]
    
    # Build the output DataFrame
    result = pd.DataFrame({
        "Embeddings": unique_embeddings,
        "PreNums": [pre_counts[i] for i in range(num_variants)],
        "PostNums": [post_counts[i] for i in range(num_variants)]
    })
    
    return result


In [ ]:
layers = 30

bg_path = "/Users/dylanwells/CodingProjects/popDMS/esmDMS/data/inference_data/BG505"
bf_path = "/Users/dylanwells/CodingProjects/popDMS/esmDMS/data/inference_results"

bg_all_layer_embeds = []
bf_all_layer_embeds = []
whole_bg_df = pd.DataFrame()
whole_bf_df = pd.DataFrame()


for layer in range(layers+1):
    print(f"Processing layer {layer}...")
    bf_layer_df = pickle.load(open(f"{bf_path}/layer{layer}/inference_df.pkl", 'rb'))
    bg_layer_df = pickle.load(open(f"{bg_path}/layer{layer}/inference_df.pkl", 'rb'))
    
    formatted_bf_df = inverse_embedding_df_transfer(bf_layer_df)
    formatted_bg_df = inverse_embedding_df_transfer(bg_layer_df)
    
    
    bf_embeddings = np.array([x for x in formatted_bf_df["Embeddings"].to_list()])
    bg_embeddings = np.array([x for x in formatted_bg_df["Embeddings"].to_list()])
    
    dimensions = bf_embeddings.shape[1]
    for dim in range(dimensions):
        bf_embeddings[:, dim] = z_normalize(bf_embeddings[:, dim])
        bg_embeddings[:, dim] = z_normalize(bg_embeddings[:, dim])
    
    formatted_bf_df["Embeddings"] = [bf_embeddings[i] for i in range(bf_embeddings.shape[0])]
    formatted_bg_df["Embeddings"] = [bg_embeddings[i] for i in range(bg_embeddings.shape[0])]
    
    
    if layer == 0:
        whole_bg_df["PreNums"] = formatted_bg_df["PreNums"]
        whole_bg_df["PostNums"] = formatted_bg_df["PostNums"]
        whole_bf_df["PreNums"] = formatted_bf_df["PreNums"]
        whole_bf_df["PostNums"] = formatted_bf_df["PostNums"]
        
        # put the first layer embeddings in a list
        bg_all_layer_embeds = [bg_embeddings]
        bf_all_layer_embeds = [bf_embeddings]
    else:
        # append the layer embeddings to the list
        bg_all_layer_embeds.append(bg_embeddings)
        bf_all_layer_embeds.append(bf_embeddings)

# make a full copy of bg_all_layer_embeds and bf_all_layer_embeds 
import copy
bg_all_layer_embeds_copy = copy.deepcopy(bg_all_layer_embeds)
bf_all_layer_embeds_copy = copy.deepcopy(bf_all_layer_embeds)

In [ ]:
bg_path = "/Users/dylanwells/CodingProjects/popDMS/esmDMS/data/inference_data/BG505"
bf_path = "/Users/dylanwells/CodingProjects/popDMS/esmDMS/data/inference_results"

In [ ]:
bg_inference_data = analyze_layers_piecewise(in_path=bg_path, verbose=False)

In [ ]:
bf_inference_data = analyze_layers_piecewise(in_path=bf_path, verbose=False)

In [ ]:
# Let's plot this 

def make_grfp_plots_inf(inference_data, normalize=True):
    """Make GRFP plots for the dataframe"""
    # inference_data = [dx, icov, s, s_joint, sel_data, gamma_opt, x_array] for layer
    # inference_data = analyze_layers(df, verbose=False)
    selection_coeffs = []
    for layer in range(len(inference_data)):
        s = inference_data[layer][2]
        #print(f"Selection coefficients for layer {layer}: {s}")
        selection_coeffs.append(s)
        
    num_reps = len(selection_coeffs[0])
    rep_combs = []
    for i in range(num_reps):
        for j in range(i+1, num_reps):
            rep_combs.append((i, j))
    print(f"rep_combs: {rep_combs}")
    num_combs = len(rep_combs)
    # Make a figure with subplots for each replicate combination
    fig, axs = plt.subplots(1, num_combs, figsize=(6*num_combs, 6))
    for comb_index, (rep_i, rep_j) in enumerate(rep_combs):
        ax = axs[comb_index]
        for layer in range(len(selection_coeffs)):
            s = selection_coeffs[layer].copy()
            if normalize:
                s[rep_i] = s[rep_i] / np.max(np.abs(s[rep_i]))
                s[rep_j] = s[rep_j] / np.max(np.abs(s[rep_j]))
            
            ax.scatter(s[rep_i], s[rep_j], label=f'Layer {layer}')
            
        ax.set_title(f'Replicate {rep_i+1} vs Replicate {rep_j+1}')
        ax.set_xlabel(f'Selection Coefficients Replicate {rep_i+1}')
        ax.set_ylabel(f'Selection Coefficients Replicate {rep_j+1}')
        ax.axis('square')
        #ax.legend()
    plt.style.use('seaborn-v0_8-darkgrid')
    plt.suptitle('Replicate Consistency Plots Across Layers', fontsize=16)
    plt.show()


def get_correlations(selection_data):
    """Get the pearson correlation data from the selection data"""
    # Data format:
    # s = [[s_rep_1_layer_1, s_rep_2_layer_1, s_rep_3_layer_1], 
    #     [s_rep_1_layer_2, s_rep_2_layer_2, s_rep_3_layer_2], ...]
    
    num_layers = len(selection_data)
    num_reps = len(selection_data[0])
    rep_combs = []
    for i in range(num_reps):
        for j in range(i+1, num_reps):
            rep_combs.append((i, j))
            
    all_corrs = []
    for layer in range(num_layers):
        s = selection_data[layer]
        layer_corrs = []
        for (rep_i, rep_j) in rep_combs:
            corr = st.pearsonr(s[rep_i], s[rep_j])[0]
            layer_corrs.append(corr)
        all_corrs.append(layer_corrs)
    
    return np.array(all_corrs)  # shape: (num_layers, num_combs)

In [ ]:
selection_coeffs_bg = []
selection_coeffs_bf = []
for layer in range(len(bg_inference_data)):
    s_bg = bg_inference_data[layer][2]
    s_bf = bf_inference_data[layer][2]
    selection_coeffs_bg.append(s_bg)
    selection_coeffs_bf.append(s_bf)


def get_sel_rep_i(layer, i):
    if i < len(selection_coeffs_bg[0]):
        return selection_coeffs_bf[layer][i]
    else:
        return selection_coeffs_bg[layer][i - len(selection_coeffs_bg[0])]


num_reps = len(selection_coeffs_bg[0])
rep_combs = []
total_reps = num_reps * 2
for i in range(total_reps):
    for j in range(i+1, total_reps):
        rep_combs.append((i, j))
num_combs = len(rep_combs)
print(f"rep_combs: {rep_combs}")

# Square, diagonal plots
fig, axs = plt.subplots(total_reps, total_reps, figsize=(4*total_reps, 4*total_reps))
for i in range(total_reps):
    for j in range(total_reps):
        ax = axs[i, j]
        """if i == j:
            
            
        else:"""
        
        
        best_pearson_r = -2
        total_pearson_r = 0
        
        
        for layer in range(len(selection_coeffs_bg)):
            s_i = get_sel_rep_i(layer, i)
            s_j = get_sel_rep_i(layer, j)
            
            s_i = s_i / np.max(np.abs(s_i))
            s_j = s_j / np.max(np.abs(s_j))
            
            pearson_r = st.pearsonr(s_i, s_j)[0]
            total_pearson_r += pearson_r
            if pearson_r > best_pearson_r:
                best_pearson_r = pearson_r
            
            
            ax.scatter(s_i, s_j, label=f'Layer {layer}', alpha=0.6)
        avg_pearson_r = total_pearson_r / len(both_data)
        ax.set_title(f'Best r: {best_pearson_r:.2f}, Avg r: {avg_pearson_r:.2f}', fontsize=12)
        
        #ax.set_xlabel(f'Select. Coeff. Rep {i+1}')
        #ax.set_ylabel(f'Select. Coeff. Rep {j+1}')
        ax.axis('square')
plt.style.use('seaborn-v0_8-darkgrid')
plt.suptitle('Replicate Consistency Plots Across Layers (BG505 + BF520)', fontsize=48,
                y=0.92)

# Label the X and Y axes of the outer plots


rep_to_label = {
    1: 'BF Rep 1',
    2: 'BF Rep 2',
    3: 'BF Rep 3',
    4: 'BG Rep 1',
    5: 'BG Rep 2',
    6: 'BG Rep 3'
}

for i in range(total_reps):
    axs[total_reps-1, i].set_xlabel(rep_to_label[i+1], fontsize=24)
    axs[i, 0].set_ylabel(rep_to_label[i+1], fontsize=24)

# change the font of the plot to latex and bold
plt.rcParams.update({'font.family': 'serif',
                    'text.usetex': True,
                    'font.weight': 'bold'})
                    
                    


plt.show()


# Let's do this but combine inference with both at once

In [ ]:
# reload esmdmsfunctions import
from esmdmsfunctions import *

In [ ]:
bg_path = "/Users/dylanwells/CodingProjects/popDMS/esmDMS/data/inference_data/BG505"
bf_path = "/Users/dylanwells/CodingProjects/popDMS/esmDMS/data/inference_results"

in_paths = [bf_path, bg_path]
    

both_data = analyze_layers_cross_variant(in_paths=in_paths, verbose=True)

In [ ]:
bg_path = "/Users/dylanwells/CodingProjects/popDMS/esmDMS/data/inference_data/BG505"
bf_path = "/Users/dylanwells/CodingProjects/popDMS/esmDMS/data/inference_results"

in_paths = [bf_path]
    

both_data = analyze_layers_cross_variant(in_paths=in_paths, verbose=True)

In [ ]:
bg_path = "/Users/dylanwells/CodingProjects/popDMS/esmDMS/data/inference_data/BG505"
bf_path = "/Users/dylanwells/CodingProjects/popDMS/esmDMS/data/inference_results"

in_paths = [bg_path]
    

both_data = analyze_layers_cross_variant(in_paths=in_paths, verbose=True)

In [ ]:
"""selection_coeffs_bg = []
selection_coeffs_bf = []
for layer in range(len(bg_inference_data)):
    s_bg = bg_inference_data[layer][2]
    s_bf = bf_inference_data[layer][2]
    selection_coeffs_bg.append(s_bg)
    selection_coeffs_bf.append(s_bf)




def get_sel_rep_i(layer, i):
    if i < len(selection_coeffs_bg[0]):
        return selection_coeffs_bg[layer][i]
    else:
        return selection_coeffs_bf[layer][i - len(selection_coeffs_bg[0])]"""


num_reps = 6
rep_combs = []
for i in range(num_reps):
    for j in range(i+1, num_reps):
        rep_combs.append((i, j))
num_combs = len(rep_combs)
print(f"rep_combs: {rep_combs}")

# Square, diagonal plots
fig, axs = plt.subplots(num_reps, num_reps, figsize=(4*num_reps, 4*num_reps))
for i in range(num_reps):
    for j in range(num_reps):
        ax = axs[i, j]
        """if i == j:
            
            
        else:"""
        
        best_pearson_r = -2
        total_pearson_r = 0
        
        
        for layer in range(len(both_data)):
            s_i = both_data[layer][2][i]
            s_j = both_data[layer][2][j]
            
            s_i = s_i / np.max(np.abs(s_i))
            s_j = s_j / np.max(np.abs(s_j))
            
            pearson_r = st.pearsonr(s_i, s_j)[0]
            total_pearson_r += pearson_r
            if pearson_r > best_pearson_r:
                best_pearson_r = pearson_r
            
            ax.scatter(s_i, s_j, label=f'Layer {layer}', alpha=0.6)
        avg_pearson_r = total_pearson_r / len(both_data)
        ax.set_title(f'Best r: {best_pearson_r:.2f}, Avg r: {avg_pearson_r:.2f}', fontsize=12)
            
        #ax.set_xlabel(f'Select. Coeff. Rep {i+1}')
        #ax.set_ylabel(f'Select. Coeff. Rep {j+1}')
        ax.axis('square')
plt.style.use('seaborn-v0_8-darkgrid')
plt.suptitle('Replicate Consistency Plots Across Layers (BG505 + BF520)', fontsize=48,
                y=0.92)

# Label the X and Y axes of the outer plots


rep_to_label = {
    1: 'BF Rep 1',
    2: 'BF Rep 2',
    3: 'BF Rep 3',
    4: 'BG Rep 1',
    5: 'BG Rep 2',
    6: 'BG Rep 3'
}

for i in range(num_reps):
    axs[num_reps-1, i].set_xlabel(rep_to_label[i+1], fontsize=24)
    axs[i, 0].set_ylabel(rep_to_label[i+1], fontsize=24)

# change the font of the plot to latex and bold
plt.rcParams.update({'font.family': 'serif',
                    'text.usetex': True,
                    'font.weight': 'bold'})
                    
plt.show()


## Layer vs Correlation for these plots

In [ ]:
num_reps = 6
rep_combs = []
for i in range(num_reps):
    for j in range(i+1, num_reps):
        rep_combs.append((i, j))
num_combs = len(rep_combs)
print(f"rep_combs: {rep_combs}")

pearson_r_dict = {}

for i in range(num_reps):
    for j in range(i+1, num_reps):
        if i == j:
            continue
        else:
            best_pearson_r = -2
            total_pearson_r = 0
            
            for layer in range(len(both_data)):
                s_i = both_data[layer][2][i]
                s_j = both_data[layer][2][j]
                
                s_i = s_i / np.max(np.abs(s_i))
                s_j = s_j / np.max(np.abs(s_j))
                
                pearson_r = st.pearsonr(s_i, s_j)[0]
                
                
                pearson_r_dict[(i, j, layer)] = pearson_r
                
                total_pearson_r += pearson_r
                if pearson_r > best_pearson_r:
                    best_pearson_r = pearson_r
                
            avg_pearson_r = total_pearson_r / len(both_data)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
n_layers = len(both_data)

rep_to_label = {
    1: 'BF Rep 1',
    2: 'BF Rep 2',
    3: 'BF Rep 3',
    4: 'BG Rep 1',
    5: 'BG Rep 2',
    6: 'BG Rep 3'
}
colors_list = sns.color_palette("tab10", n_colors=10)
textures_list = ['-', '--', '-.', ':', (0, (3, 1, 1, 1)), (0, (5, 1))]

plt.style.use('fivethirtyeight')
plt.figure(figsize=(12, 8))
idx = 0
for (i, j) in rep_combs:
    if i == j:
        continue
    pearson_rs = [pearson_r_dict[(i, j, layer)] for layer in range(n_layers)]
    plt.plot(range(n_layers), pearson_rs, marker='o',
             label=f'{rep_to_label[i+1]} vs {rep_to_label[j+1]}',
             color=colors_list[idx % len(colors_list)],
             linestyle=textures_list[idx % len(textures_list)],
                linewidth=2, markersize=8)
    idx += 1
plt.xlabel('Layer')
plt.ylabel('Pearson r')
plt.title('Pearson Correlation of Selection Coefficients Across Layers')
plt.legend()
plt.grid(True)
plt.show()



### Now invert the plot

above is the Pearson_r across layer for each dimension. Now, I want the layer to be the color and the experimental comparison to be the x-axis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

n_layers = len(both_data)
rep_to_label = {
    1: 'BF Rep 1',
    2: 'BF Rep 2',
    3: 'BF Rep 3',
    4: 'BG Rep 1',
    5: 'BG Rep 2',
    6: 'BG Rep 3'
}

# Build list of valid rep combos and their labels
valid_combs = [(i, j) for (i, j) in rep_combs if i != j]
x_labels = [f'{rep_to_label[i+1]} vs {rep_to_label[j+1]}' for (i, j) in valid_combs]
x_positions = range(len(valid_combs))

# 31 layers need more than 10 colors — use a continuous colormap
#colors_list = sns.color_palette("husl", n_colors=n_layers)
colors_list = [plt.cm.Spectral(x) for x in np.linspace(0, 1, n_layers)]
textures_list = ['-', '--', '-.', ':', (0, (3, 1, 1, 1)), (0, (5, 1))]

plt.style.use('ggplot')
plt.figure(figsize=(16, 8))

for layer in range(n_layers):
    pearson_rs = [pearson_r_dict[(i, j, layer)] for (i, j) in valid_combs]
    plt.plot(x_positions, pearson_rs, marker='o',
             label=f'Layer {layer}',
             color=colors_list[layer % len(colors_list)],
             linestyle=textures_list[layer % len(textures_list)],
             linewidth=2, markersize=6)

plt.xticks(list(x_positions), x_labels, rotation=45, ha='right')
plt.xlabel('Replicate Combination')
plt.ylabel('Pearson r')
plt.title('Pearson Correlation of Selection Coefficients Across Replicate Combinations')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small', ncol=2)
plt.grid(True)
plt.tight_layout()
plt.show()

# Covariance between embeddings of BG505 and BF520

In [ ]:
bg_path = "/Users/dylanwells/CodingProjects/popDMS/esmDMS/data/inference_data/BG505"
bf_path = "/Users/dylanwells/CodingProjects/popDMS/esmDMS/data/inference_results"

In [ ]:
num_layers = 31
bg_layer_corrs = np.zeros((num_layers, 3))  # 3 replicate combinations
bf_layer_corrs = np.zeros((num_layers, 3))  # 3 replicate combinations
for layer in range(num_layers):
    print(f"Processing layer {layer}...")
    bf_layer_df = pickle.load(open(f"{bf_path}/layer{layer}/inference_df.pkl", 'rb'))
    bg_layer_df = pickle.load(open(f"{bg_path}/layer{layer}/inference_df.pkl", 'rb'))
    
    bf_embeddings = np.array([x for x in bf_layer_df["Embedding"].to_list()])
    bg_embeddings = np.array([x for x in bg_layer_df["Embedding"].to_list()])
    
    dimensions = bf_embeddings.shape[1]
    for dim in range(dimensions):
        bf_embeddings[:, dim] = z_normalize(bf_embeddings[:, dim])
        bg_embeddings[:, dim] = z_normalize(bg_embeddings[:, dim])
        
    # Now find the covariance between dimensions for each replicate pair
    cov_bg = np.cov(bg_embeddings, rowvar=False)
    cov_bf = np.cov(bf_embeddings, rowvar=False)
    
    bg_corr_matrix = np.corrcoef(bg_embeddings, rowvar=False)
    bf_corr_matrix = np.corrcoef(bf_embeddings, rowvar=False)
    
    # Plot the covariance matrices
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.title(f'BG505 Layer {layer} Covariance Matrix')
    plt.imshow(cov_bg, cmap='viridis')
    plt.colorbar()
    plt.subplot(1, 2, 2)
    plt.title(f'BF520 Layer {layer} Covariance Matrix')
    plt.imshow(cov_bf, cmap='viridis')
    plt.colorbar()
    plt.show()
    
    # Plot the correlation matrices
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.title(f'BG505 Layer {layer} Correlation Matrix')
    plt.imshow(bg_corr_matrix, cmap='viridis', vmin=-1, vmax=1)
    plt.colorbar()
    plt.subplot(1, 2, 2)
    plt.title(f'BF520 Layer {layer} Correlation Matrix')
    plt.imshow(bf_corr_matrix, cmap='viridis', vmin=-1, vmax=1)
    plt.colorbar()
    plt.show()
    
    

    
    

In [ ]:
# Let's try some different visualizations of the correlation matrices here. 
chosen_layer = 30
bf_layer_df = pickle.load(open(f"{bf_path}/layer{chosen_layer}/inference_df.pkl", 'rb'))
bf_embeddings = np.array([x for x in bf_layer_df["Embedding"].to_list()])

bg_layer_df = pickle.load(open(f"{bg_path}/layer{chosen_layer}/inference_df.pkl", 'rb'))
bg_embeddings = np.array([x for x in bg_layer_df["Embedding"].to_list()])

dimensions = bf_embeddings.shape[1]
for dim in range(dimensions):
    bf_embeddings[:, dim] = z_normalize(bf_embeddings[:, dim])
    bg_embeddings[:, dim] = z_normalize(bg_embeddings[:, dim])

# Now find the covariance between dimensions for each replicate pair
cov_bg = np.cov(bg_embeddings, rowvar=False)
cov_bf = np.cov(bf_embeddings, rowvar=False)
bg_corr_matrix = np.corrcoef(bg_embeddings, rowvar=False)
bf_corr_matrix = np.corrcoef(bf_embeddings, rowvar=False)

import seaborn as sns
plt.figure(figsize=(10, 10))

"""plt.subplot(1, 2, 1)"""
# This clusters and reorders automatically
g = sns.clustermap(cov_bf, cmap='RdBu_r', center=0,
                   vmin=-1, vmax=1, figsize=(12, 12))

plt.title(f'BF520 Layer {chosen_layer} Correlation Matrix (Clustered)', fontsize=16)
"""plt.subplot(1, 2, 2)"""
"""plt.title(f'BG505 Layer {chosen_layer} Correlation Matrix (Clustered)', fontsize=16)
g = sns.clustermap(cov_bg, cmap='RdBu_r', center=0,
                   vmin=-1, vmax=1, figsize=(12, 12))"""
plt.show()

import seaborn as sns
plt.figure(figsize=(10, 10))
"""plt.subplot(1, 2, 1)"""
# This clusters and reorders automatically
g = sns.clustermap(bf_corr_matrix, cmap='RdBu_r', center=0,
                   vmin=-1, vmax=1, figsize=(12, 12))
plt.title(f'BF520 Layer {chosen_layer} Correlation Matrix (Clustered)', fontsize=16)
"""plt.subplot(1, 2, 2)"""
"""plt.title(f'BG505 Layer {chosen_layer} Correlation Matrix (Clustered)', fontsize=16)
g = sns.clustermap(bg_corr_matrix, cmap='RdBu_r', center=0,
                   vmin=-1, vmax=1, figsize=(12, 12))"""
plt.show()




In [ ]:
# Let's try some different visualizations of the correlation matrices here. 
chosen_layer = 22
bf_layer_df = pickle.load(open(f"{bf_path}/layer{chosen_layer}/inference_df.pkl", 'rb'))
bf_embeddings = np.array([x for x in bf_layer_df["Embedding"].to_list()])

bg_layer_df = pickle.load(open(f"{bg_path}/layer{chosen_layer}/inference_df.pkl", 'rb'))
bg_embeddings = np.array([x for x in bg_layer_df["Embedding"].to_list()])

total_embeddings = np.vstack((bf_embeddings, bg_embeddings))
total_embeddings = np.unique(total_embeddings, axis=0)
dimensions = total_embeddings.shape[1]
for dim in range(dimensions):
    total_embeddings[:, dim] = z_normalize(total_embeddings[:, dim])
    
cov_total = np.cov(total_embeddings, rowvar=False)
corr_matrix_total = np.corrcoef(total_embeddings, rowvar=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

matrices = [corr_matrix_total, bg_corr_matrix, bf_corr_matrix]
titles = ['Total', 'BG', 'BF']

for ax, matrix, title in zip(axes, matrices, titles):
    im = ax.imshow(matrix, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_title(f'{title} Layer {chosen_layer} Correlation Matrix', fontsize=14)
    fig.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

matrices = [cov_total, cov_bg, cov_bf]
titles = ['Total', 'BG', 'BF']

for ax, matrix, title in zip(axes, matrices, titles):
    im = ax.imshow(matrix, cmap='RdBu_r', vmin=-1, vmax=1)
    # set title to bold
    ax.set_title(f'{title} Layer {chosen_layer} Covariance Matrix', fontsize=20, 
                    fontweight='bold')
    fig.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
# This clusters and reorders automatically
g = sns.clustermap(corr_matrix_total, cmap='RdBu_r', center=0,
                   vmin=-1, vmax=1, figsize=(12, 12))
                   
plt.title(f'Total Layer {chosen_layer} Correlation Matrix', fontsize=16)
plt.show()



In [ ]:
import networkx as nx
#import matplotlib.pyplot as plt

threshold = 0.7
G = nx.Graph()

for i in range(640):
    for j in range(i+1, 640):
        if abs(corr_matrix_total[i, j]) > threshold:
            G.add_edge(i, j, weight=corr_matrix_total[i, j])

# Remove isolated nodes
G.remove_nodes_from(list(nx.isolates(G)))

plt.figure(figsize=(12, 12))
pos = nx.spring_layout(G, k=0.5)
nx.draw(G, pos, node_size=20, width=0.5, alpha=0.7)
plt.title(f"Dimensions with |r| > {threshold}")

In [ ]:
cov_mats = [cov_bf, cov_bg, cov_total]
titles = ['BF Covariance Matrix', 'BG Covariance Matrix', 'Total Covariance Matrix']

"""import matplotlib.pyplot as plt

# Extract upper triangle (excluding diagonal)
upper = corr_matrix[np.triu_indices(660, k=1)]

plt.hist(upper, bins=100, edgecolor='none')
plt.xlabel("Correlation")
plt.ylabel("Count")
plt.title("Distribution of pairwise correlations")"""


colors = sns.color_palette("husl", 3)

plt.figure(figsize=(15, 5))
for i, cov_mat in enumerate(cov_mats):
    plt.subplot(1, 3, i+1)
    upper = cov_mat[np.triu_indices(cov_mat.shape[0], k=1)]
    plt.hist(upper, bins=100, edgecolor='none', color=colors[i])
    plt.xlabel("Covariance")
    plt.ylabel("Count")
    plt.xlim([-1.0, 1.0])
    plt.title(titles[i])
    
plt.tight_layout()
plt.show()

In [ ]:
"""import networkx as nx
#import matplotlib.pyplot as plt

threshold = 0.7
G = nx.Graph()

for i in range(640):
    for j in range(i+1, 640):
        if abs(corr_matrix_total[i, j]) > threshold:
            G.add_edge(i, j, weight=corr_matrix_total[i, j])

# Remove isolated nodes
G.remove_nodes_from(list(nx.isolates(G)))

plt.figure(figsize=(12, 12))
pos = nx.spring_layout(G, k=0.5)
nx.draw(G, pos, node_size=20, width=0.5, alpha=0.7)
plt.title(f"Dimensions with |r| > {threshold}")"""

cov_mats = [cov_bf, cov_bg, cov_total]
titles = ['BF Covariance Matrix', 'BG Covariance Matrix', 'Total Covariance Matrix']

plt.figure(figsize=(15, 5))
for i, cov_mat in enumerate(cov_mats):
    plt.subplot(1, 3, i+1)
    G = nx.Graph()
    threshold = 0.7
    for m in range(cov_mat.shape[0]):
        for n in range(m+1, cov_mat.shape[0]):
            if abs(cov_mat[m, n]) > threshold:
                G.add_edge(m, n, weight=cov_mat[m, n])
    G.remove_nodes_from(list(nx.isolates(G)))
    pos = nx.spring_layout(G, k=0.5)
    nx.draw(G, pos, node_size=20, width=0.5, alpha=0.7)
    plt.title(titles[i])
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.decomposition import PCA

cov_mats = [cov_bf, cov_bg, cov_total]
titles = ['BF Covariance Matrix', 'BG Covariance Matrix', 'Total Covariance Matrix']

"""pca = PCA()
pca.fit(X)  # or use your weighted version

# How many components explain most variance?
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel("Number of components")
plt.ylabel("Cumulative variance explained")"""

plt.figure(figsize=(15, 5))
for i, cov_mat in enumerate(cov_mats):
    plt.subplot(1, 3, i+1)
    pca = PCA()
    pca.fit(cov_mat)
    plt.plot(np.cumsum(pca.explained_variance_ratio_), color=colors[i])
    plt.xlabel("Number of components")
    plt.ylabel("Cumulative variance explained")
    plt.title(titles[i])
plt.tight_layout()
plt.show()

In [ ]:
"""# Top 5 dimensions contributing to PC1
pc1_loadings = pca.components_[0]
top_dims = np.argsort(np.abs(pc1_loadings))[-5:]
print("Top dims for PC1:", top_dims)
print("Loadings:", pc1_loadings[top_dims])"""

cov_mats = [cov_bf, cov_bg, cov_total]
titles = ['BF Covariance Matrix', 'BG Covariance Matrix', 'Total Covariance Matrix']
for i, cov_mat in enumerate(cov_mats):
    pca = PCA()
    pca.fit(cov_mat)
    pc1_loadings = pca.components_[0]
    top_dims = np.argsort(np.abs(pc1_loadings))[-10:]
    print(f"Top dims for PC1 in {titles[i]}:", top_dims)
    print("Loadings:", pc1_loadings[top_dims])
    